____
# COMPUTE GRADIENT, INTERPOLATE AND ROTATE SWOT DATA ON COLOC POINTS

In [4]:
import numpy as np
import pandas as pd
import xarray as xr

import matplotlib.pyplot as plt

import os
from glob import glob

from cstes import swot_dir_2km, drifters_dir, get_proj, lonlat2xy, zarr_dir
from swot import browse_swot_2km, add_mask_inside_swot, build_swath_polygon

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.geodesic as cgeo
crs = ccrs.PlateCarree()

import cartopy.geodesic as geod
import cartopy.crs as ccrs
import cartopy.feature as cfeature

import pyproj
from pyproj import Geod

from rasterio.transform import Affine

import pynsitu as pyn

_______
# Add delay on the cycle

In [5]:
dc = 4

In [16]:
dfs

,day,pass_number,cycle_number,file,time,start_time_cut,end_time_cut
0,2023-04-01,3,478,/Users/mdemol/DATA_KARIN/L3_calval_1.0.2/3_478...,2023-04-01 23:42:32.616767104,2023-04-01 11:47:13.685128640,2023-04-02 11:37:51.501384128
1,2023-04-02,3,479,/Users/mdemol/DATA_KARIN/L3_calval_1.0.2/3_479...,2023-04-02 23:33:10.386001152,2023-04-02 11:37:51.501384128,2023-04-03 11:28:29.256323264
2,2023-04-03,3,480,/Users/mdemol/DATA_KARIN/L3_calval_1.0.2/3_480...,2023-04-03 23:23:48.126645376,2023-04-03 11:28:29.256323264,2023-04-04 11:19:06.980127424
3,2023-04-04,3,481,/Users/mdemol/DATA_KARIN/L3_calval_1.0.2/3_481...,2023-04-04 23:14:25.833609472,2023-04-04 11:19:06.980127424,2023-04-05 11:09:44.712222784
4,2023-04-05,3,482,/Users/mdemol/DATA_KARIN/L3_calval_1.0.2/3_482...,2023-04-05 23:05:03.590836096,2023-04-05 11:09:44.712222784,2023-04-06 11:00:22.710395136
...,...,...,...,...,...,...,...
186,2023-07-05,16,573,/Users/mdemol/DATA_KARIN/L3_calval_1.0.2/16_57...,2023-07-05 19:33:50.497169792,2023-07-05 07:38:31.521460288,2023-07-06 07:29:09.458071104
187,2023-07-06,16,574,/Users/mdemol/DATA_KARIN/L3_calval_1.0.2/16_57...,2023-07-06 19:24:28.418972416,2023-07-06 07:29:09.458071104,2023-07-07 07:19:47.366957440
188,2023-07-07,16,575,/Users/mdemol/DATA_KARIN/L3_calval_1.0.2/16_57...,2023-07-07 19:15:06.314942464,2023-07-07 07:19:47.366957440,2023-07-08 07:10:25.246364480
189,2023-07-08,16,576,/Users/mdemol/DATA_KARIN/L3_calval_1.0.2/16_57...,2023-07-08 19:05:44.177786496,2023-07-08 07:10:25.246364480,2023-07-09 07:01:03.101010112


In [10]:
dfs = browse_swot_2km().reset_index()
drifters_sources = 'all_med_variational_10min_v0.nc'
#drifters_sources = 'all_med_lowess_10min_v0.nc'
df = pd.read_csv(os.path.join(zarr_dir, 'drifters_'+drifters_sources.replace('.nc', '.csv')), parse_dates=['datetime'])#.drop(columns='Unnamed: 0')
possible_cycle = np.unique(df.cycle_number)
dfr = df.copy()
dfr['cycle_number'] = (dfr.cycle_number+dc).astype(int)
dfr = dfr[dfr.cycle_number.isin(possible_cycle)].drop(columns=['cycle_date', 'time_to_swot'])

/var/folders/fn/z858c2qj1lz65xr0z5mdvbf40000gp/T/ipykernel_49039/1824984786.py:4: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(os.path.join(zarr_dir, 'drifters_'+drifters_sources.replace('.nc', '.csv')), parse_dates=['datetime'])#.drop(columns='Unnamed: 0')


__________
# SWOT

In [4]:
# get grid orientation and metrics
def add_grid_metrics(ds):
    """ add grid spatial metrics """

    geod = Geod(ellps="WGS84")

    lon, lat = ds.longitude, ds.latitude
    dims = lon.dims
    
    # d/dx where x is cross-track
    az12, az21, dx = geod.inv(
        lon, lat, lon.shift(num_pixels=-1), lat.shift(num_pixels=-1),
    )
    
    ds = ds.assign_coords(dx=(dims, dx), phi=(dims, az12*np.pi/180))
    
    ds["dx"] = (
        ds["dx"]
        .ffill("num_pixels")
        .where(ds["duacs_editing_flag"]<5)
    )

    # phi_lon is cross-track direction from north
    ds["phix"] = (
        ds["phi"]
        .ffill("num_pixels")
        .where(ds["duacs_editing_flag"]<5)
    )

    # d/dy where y is along-track
    az12, az21, dy = geod.inv(
        lon, lat, lon.shift(num_lines=-1), lat.shift(num_lines=-1),
    )
    
    # phiy is along-track direction from north
    ds = ds.assign_coords(phi=(dims, az12*np.pi/180))
    ds["phiy"] = (
        ds["phi"]
        .ffill("num_pixels")
        .where(ds["duacs_editing_flag"]<5)
    )
    
    ds = ds.assign_coords(dy=(dims, dy))
    
    ds["dy"] = (
        ds["dy"]
        .ffill("num_lines")
        .where(ds["duacs_editing_flag"]<5)
    )
    ds['phi']=ds['phix']
    return ds

In [5]:
cycle = 500
swath = 3

dss = xr.open_dataset(dfs.where((dfs.pass_number==swath)&(dfs.cycle_number==cycle)).dropna().file.values[0])
dss= add_grid_metrics(dss)
#list(dss.keys())

/Users/mdemol/opt/anaconda3/envs/equinox/lib/python3.10/site-packages/xarray/backends/plugins.py:159: RuntimeWarning: 'netcdf4' fails while guessing
  warnings.warn(f"{engine!r} fails while guessing", RuntimeWarning)
/Users/mdemol/opt/anaconda3/envs/equinox/lib/python3.10/site-packages/xarray/backends/plugins.py:159: RuntimeWarning: 'scipy' fails while guessing
  warnings.warn(f"{engine!r} fails while guessing", RuntimeWarning)


In [6]:
dss

<xarray.Dataset> Size: 13MB
Dimensions:                                  (num_lines: 496, num_pixels: 69)
Coordinates:
    latitude                                 (num_lines, num_pixels) float64 274kB ...
    latitude_nadir                           (num_lines) float64 4kB ...
    longitude                                (num_lines, num_pixels) float64 274kB ...
    longitude_nadir                          (num_lines) float64 4kB ...
    time                                     (num_lines) datetime64[ns] 4kB ...
    dx                                       (num_lines, num_pixels) float64 274kB ...
    phi                                      (num_lines, num_pixels) float64 274kB ...
    dy                                       (num_lines, num_pixels) float64 274kB ...
Dimensions without coordinates: num_lines, num_pixels
Data variables: (12/64)
    ancillary_surface_classification_flag    (num_lines, num_pixels) float32 137kB ...
    cross_track_distance                     (num_lines, num_pixels) float32 137kB ...
    cvl_beta_angle                           (num_lines) float32 2kB ...
    cvl_flag_val                             (num_lines, num_pixels) float32 137kB ...
    cvl_flag_val_outliers                    (num_lines, num_pixels) float32 137kB ...
    cvl_mean_dynamic_topography_cnes_cls_22  (num_lines, num_pixels) float64 274kB ...
    ...                                       ...
    ssha_karin_2_qual                        (num_lines, num_pixels) float64 274kB ...
    swh_model                                (num_lines, num_pixels) float64 274kB ...
    time_tai                                 (num_lines) datetime64[ns] 4kB ...
    version                                  (num_lines) |S7 3kB ...
    phix                                     (num_lines, num_pixels) float64 274kB ...
    phiy                                     (num_lines, num_pixels) float64 274kB ...
Attributes:
    latc:     39.961324
    lonc:     4.3749
    phi:      76.80931729750104

In [7]:
dss_nan = xr.full_like(dss, np.nan)

/Users/mdemol/opt/anaconda3/envs/equinox/lib/python3.10/site-packages/numpy/core/numeric.py:407: RuntimeWarning: invalid value encountered in cast
  multiarray.copyto(res, fill_value, casting='unsafe')


In [8]:
ggd_variables = ['cvl_mean_dynamic_topography_cnes_cls_22',
                'distance_to_coast',
                 'cvl_mean_sea_surface_cnes_22_hybrid',
                 'cvl_ocean_tide_fes_2022',
                 'cvl_ssha_reference',
                 'duacs_ssha_karin_2_calibrated',
                 'duacs_ssha_karin_2_filtered',
                'duacs_ssha_karin_2_oi',
                 'internal_tide_hret',
                 'mean_sea_surface_cnescls',
                 'ocean_tide_fes',
                 'ssh_karin_uncert',
                  ]

variables = ['ancillary_surface_classification_flag',
             'cross_track_distance',
             'duacs_editing_flag',
             'duacs_phase_screen',
             'duacs_phase_screen_orbit',
             'duacs_phase_screen_static',
             'duacs_relative_vorticity',
             'duacs_speed_meridional',
             'duacs_speed_meridional_abs',
             'duacs_speed_zonal',
             'duacs_speed_zonal_abs',
             'duacs_strain',
             'duacs_xcal',
             'sig0_karin_2',
             'phi',]

# Compute gradient

## Naive way

In [9]:
# Compute ggd with "naive" approach, noise will have an impact
g = 9.80665

def gradient_naive(dss, ggd_variables=None):
    if ggd_variables : dss_ggd = dss[ggd_variables]
    dx = dss_ggd.dx.mean() # meters
    dy = dss_ggd.dy.mean()
    
    dss_ggdx = (g*dss_ggd.differentiate("num_pixels")/dx).rename({v:'ggdx_'+v for v in dss_ggd}).where(~dss['duacs_ssha_karin_2_filtered'].isnull())
    dss_ggdy = (g*dss_ggd.differentiate("num_lines")/dy).rename({v:'ggdy_'+v for v in dss_ggd}).where(~dss['duacs_ssha_karin_2_filtered'].isnull())
    return xr.merge([dss_ggdx, dss_ggdy])

dss_naive = gradient_naive(dss, ggd_variables).sel(num_lines=slice(90, 480))

In [10]:
dss_naive

<xarray.Dataset> Size: 6MB
Dimensions:                                       (num_lines: 390,
                                                   num_pixels: 69)
Coordinates:
    latitude                                      (num_lines, num_pixels) float64 215kB ...
    latitude_nadir                                (num_lines) float64 3kB 36....
    longitude                                     (num_lines, num_pixels) float64 215kB ...
    longitude_nadir                               (num_lines) float64 3kB 3.4...
    time                                          (num_lines) datetime64[ns] 3kB ...
    dx                                            (num_lines, num_pixels) float64 215kB ...
    phi                                           (num_lines, num_pixels) float64 215kB ...
    dy                                            (num_lines, num_pixels) float64 215kB ...
Dimensions without coordinates: num_lines, num_pixels
Data variables: (12/24)
    ggdx_cvl_mean_dynamic_topography_cnes_cls_22  (num_lines, num_pixels) float64 215kB ...
    ggdx_distance_to_coast                        (num_lines, num_pixels) float64 215kB ...
    ggdx_cvl_mean_sea_surface_cnes_22_hybrid      (num_lines, num_pixels) float64 215kB ...
    ggdx_cvl_ocean_tide_fes_2022                  (num_lines, num_pixels) float64 215kB ...
    ggdx_cvl_ssha_reference                       (num_lines, num_pixels) float64 215kB ...
    ggdx_duacs_ssha_karin_2_calibrated            (num_lines, num_pixels) float64 215kB ...
    ...                                            ...
    ggdy_duacs_ssha_karin_2_filtered              (num_lines, num_pixels) float64 215kB ...
    ggdy_duacs_ssha_karin_2_oi                    (num_lines, num_pixels) float64 215kB ...
    ggdy_internal_tide_hret                       (num_lines, num_pixels) float64 215kB ...
    ggdy_mean_sea_surface_cnescls                 (num_lines, num_pixels) float64 215kB ...
    ggdy_ocean_tide_fes                           (num_lines, num_pixels) float64 215kB ...
    ggdy_ssh_karin_uncert                         (num_lines, num_pixels) float64 215kB ...

# Interpolate 

In [11]:
#dfr = pd.read_csv(os.path.join(zarr_dir, 'drifters_'+drifters_sources.replace('.nc', '.csv')))#.drop(columns = 'Unnamed: 0').set_index('row_number')

In [12]:
#dfr_ = dfr.where((dfr.pass_number==swath)&(dfr.cycle_number==cycle)).dropna()

In [14]:
import pyinterp
mesh = pyinterp.RTree()

def interp_one_dataarray(da, new_lon, new_lat):
    lons = da.longitude.compute().data.flatten()
    lats = da.latitude.compute().data.flatten()
    data = da.compute().data.flatten()

    mesh.packing(np.vstack((lons, lats)).T, data)
    mx = new_lon
    my = new_lat

    idw, neighbors = mesh.inverse_distance_weighting(
        np.vstack((mx.ravel(), my.ravel())).T,
        within=False,  # Extrapolation is forbidden
        k=5,  # We are looking for at most 11 neighbors
        num_threads=0,
    )
    idw = idw.reshape(mx.shape)
    return idw

def interp_dss(dss, new_lon, new_lat):
    df_interp = pd.DataFrame()
    df_interp['longitude'] = new_lon
    df_interp['latitude'] = new_lat
    for v in dss : 
        da = dss[v]
        df_interp[v]=interp_one_dataarray(da, new_lon, new_lat)
    return df_interp
        
        

In [15]:
pd.to_datetime(dss.time.values[0])

Timestamp('2023-04-23 20:16:27.384615552')

____________
# Compute and interp at once

In [16]:
def coloc_swot_cycle_swath(dfr, dfs, cycle, swath, cutoff, method_gradient='gauss'):
    # select drifter point
    dfr_ = dfr.where((dfr.pass_number==swath)&(dfr.cycle_number==cycle)).dropna()
    try : 
        dss = xr.open_dataset(dfs.where((dfs.pass_number==swath)&(dfs.cycle_number==cycle)).dropna().file.values[0])
    except : 
        dss = dss_nan
        print(swath, cycle)
    dss = add_grid_metrics(dss).reset_coords(['phi'])
    if method_gradient =='naive' :
        dss_ggd= gradient_naive(dss, ggd_variables=ggd_variables)
    if method_gradient == 'gauss' :
        dss_ggd = gradient_gauss(dss, cutoff, ggd_variables)
    swot_time = pd.to_datetime(dss.time.values[0])
    dss = xr.merge([dss[variables], dss_ggd])
    df_interp = interp_dss(dss, dfr_.longitude.values, dfr_.latitude.values)
    df_out = pd.concat([dfr_.reset_index()[['row_number', 'longitude']].set_index('longitude'), df_interp.set_index('longitude')], axis=1).reset_index().set_index('row_number')
    rotate_ggd(df_out)
    df_out['time_to_swot_update']=(dfr_.datetime-swot_time).abs()
    return df_out

def coloc_swot(dfr, dfs, cutoff, method_gradient='gauss'):
    DF = []
    for swath in dfr.pass_number.unique() : 
        for cycle in dfr.where(dfr.pass_number==swath).dropna().cycle_number.unique():
            DF.append(coloc_swot_cycle_swath(dfr, dfs, cycle,  swath, cutoff, method_gradient))
    return pd.concat(DF)

# Rotate

In [17]:
def rotate(x, y, phi):
    return np.sin(phi)*x - np.cos(phi)*y, np.cos(phi)*x + np.sin(phi)*y
def rotate_ggd(df):
    for v in ggd_variables : 
        df['ggde_'+v], df['ggdn_'+v] = rotate(df['ggdx_'+v], df['ggdy_'+v], df['phi'])

__________
________
# GENERATE


In [28]:
method, cutoff = 'naive', ''
dfsg = coloc_swot(dfr, dfs, method_gradient=method, cutoff=cutoff)
rotate_ggd(dfsg)
dfsg.to_csv(os.path.join(zarr_dir, 'swot_2km_cycleplus'+str(dc)+'_'+drifters_sources.replace('.nc', '')+f'_{method}{cutoff}.csv'))


/Users/mdemol/opt/anaconda3/envs/equinox/lib/python3.10/site-packages/xarray/backends/plugins.py:159: RuntimeWarning: 'netcdf4' fails while guessing
  warnings.warn(f"{engine!r} fails while guessing", RuntimeWarning)
/Users/mdemol/opt/anaconda3/envs/equinox/lib/python3.10/site-packages/xarray/backends/plugins.py:159: RuntimeWarning: 'scipy' fails while guessing
  warnings.warn(f"{engine!r} fails while guessing", RuntimeWarning)
/Users/mdemol/opt/anaconda3/envs/equinox/lib/python3.10/site-packages/xarray/backends/plugins.py:159: RuntimeWarning: 'netcdf4' fails while guessing
  warnings.warn(f"{engine!r} fails while guessing", RuntimeWarning)
/Users/mdemol/opt/anaconda3/envs/equinox/lib/python3.10/site-packages/xarray/backends/plugins.py:159: RuntimeWarning: 'scipy' fails while guessing
  warnings.warn(f"{engine!r} fails while guessing", RuntimeWarning)
/Users/mdemol/opt/anaconda3/envs/equinox/lib/python3.10/site-packages/xarray/backends/plugins.py:159: RuntimeWarning: 'netcdf4' fails wh

3 566.0


/Users/mdemol/opt/anaconda3/envs/equinox/lib/python3.10/site-packages/xarray/backends/plugins.py:159: RuntimeWarning: 'netcdf4' fails while guessing
  warnings.warn(f"{engine!r} fails while guessing", RuntimeWarning)
/Users/mdemol/opt/anaconda3/envs/equinox/lib/python3.10/site-packages/xarray/backends/plugins.py:159: RuntimeWarning: 'scipy' fails while guessing
  warnings.warn(f"{engine!r} fails while guessing", RuntimeWarning)
/Users/mdemol/opt/anaconda3/envs/equinox/lib/python3.10/site-packages/xarray/backends/plugins.py:159: RuntimeWarning: 'netcdf4' fails while guessing
  warnings.warn(f"{engine!r} fails while guessing", RuntimeWarning)
/Users/mdemol/opt/anaconda3/envs/equinox/lib/python3.10/site-packages/xarray/backends/plugins.py:159: RuntimeWarning: 'scipy' fails while guessing
  warnings.warn(f"{engine!r} fails while guessing", RuntimeWarning)
/Users/mdemol/opt/anaconda3/envs/equinox/lib/python3.10/site-packages/xarray/backends/plugins.py:159: RuntimeWarning: 'netcdf4' fails wh

16 578.0


In [84]:
df1 = pd.read_csv(os.path.join(zarr_dir, 'swot_2km_cycleplus'+str(dc)+'_'+drifters_sources.replace('.nc', '')+f'_{method}{cutoff}.csv'))

In [85]:
list(df1.columns)

['row_number',
 'longitude',
 'latitude',
 'ancillary_surface_classification_flag',
 'cross_track_distance',
 'duacs_editing_flag',
 'duacs_phase_screen',
 'duacs_phase_screen_orbit',
 'duacs_phase_screen_static',
 'duacs_relative_vorticity',
 'duacs_speed_meridional',
 'duacs_speed_meridional_abs',
 'duacs_speed_zonal',
 'duacs_speed_zonal_abs',
 'duacs_strain',
 'duacs_xcal',
 'sig0_karin_2',
 'phi',
 'ggdx_cvl_mean_dynamic_topography_cnes_cls_22',
 'ggdx_distance_to_coast',
 'ggdx_cvl_mean_sea_surface_cnes_22_hybrid',
 'ggdx_cvl_ocean_tide_fes_2022',
 'ggdx_cvl_ssha_reference',
 'ggdx_duacs_ssha_karin_2_calibrated',
 'ggdx_duacs_ssha_karin_2_filtered',
 'ggdx_duacs_ssha_karin_2_oi',
 'ggdx_internal_tide_hret',
 'ggdx_mean_sea_surface_cnescls',
 'ggdx_ocean_tide_fes',
 'ggdx_ssh_karin_uncert',
 'ggdy_cvl_mean_dynamic_topography_cnes_cls_22',
 'ggdy_distance_to_coast',
 'ggdy_cvl_mean_sea_surface_cnes_22_hybrid',
 'ggdy_cvl_ocean_tide_fes_2022',
 'ggdy_cvl_ssha_reference',
 'ggdy_duac

In [86]:
df1.dropna().time_to_swot_update

27        1 days 17:03:48.126645376
28        1 days 16:53:48.126645376
29        1 days 16:43:48.126645376
30        1 days 16:33:48.126645376
31        1 days 16:23:48.126645376
                    ...            
376591    1 days 12:26:22.024233728
376592    1 days 12:16:22.024233728
376593    1 days 12:06:22.024233728
376594    1 days 11:56:22.024233728
376595    1 days 11:46:22.024233728
Name: time_to_swot_update, Length: 275121, dtype: object